# ChessDorrie — Tal Troll Analyzer (Colab launcher)

**How to launch:** `Runtime → Run all`. After ~1 minute the last cell prints a
public URL — open it in a new tab.

**Before you click Run all:** `Runtime → Change runtime type → T4 GPU`
(or A100/L4 if available). The GPU only matters if you want real Maia
inference via Lc0 — Stockfish runs on CPU anyway. With no GPU the bot
silently falls back to a softmax-Stockfish human-move predictor.

**Session goes idle after ~90 min** of no activity. To bring the tunnel
back: just re-run the last two cells (launch server + tunnel).

**If something looks wrong:** the very last cell tails the server log.

## 1. Clone / pull the repo

In [ ]:
import os, subprocess, sys
REPO_URL = 'https://github.com/omarnuri/chessdorrie'
REPO_BRANCH = 'claude/stoic-hypatia-U7ain'
REPO_DIR = '/content/ChessDorrie'
if not os.path.isdir(REPO_DIR):
    subprocess.run(['git', 'clone', '-b', REPO_BRANCH, REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'fetch', 'origin', REPO_BRANCH], check=False)
    subprocess.run(['git', '-C', REPO_DIR, 'checkout', REPO_BRANCH], check=False)
    subprocess.run(['git', '-C', REPO_DIR, 'pull', '--ff-only', 'origin', REPO_BRANCH], check=False)
os.chdir(REPO_DIR)
print('Working dir:', os.getcwd())
print('Branch:', subprocess.run(['git', 'rev-parse', '--abbrev-ref', 'HEAD'], capture_output=True, text=True).stdout.strip())

## 2. System dependencies (Stockfish + optional Lc0)

In [ ]:
!apt-get update -qq && apt-get install -y -qq stockfish 2>&1 | tail -3
# Install Lc0 with the CUDA backend (real GPU acceleration).
# The script resolves the latest release, downloads the linux-cuda binary,
# and runs a benchmark to confirm the GPU works.
!bash scripts/install_lc0_cuda.sh || echo "lc0 GPU install failed — softmax fallback will be used"

## 3. Python dependencies

In [ ]:
!pip install -q -r requirements.txt

## 4. Download Maia weights (1100, 1500, 1900)

Skip if you only want to use the softmax fallback.

In [ ]:
import os, urllib.request
os.makedirs('weights', exist_ok=True)
for elo in (1100, 1500, 1900):
    dest = f'weights/maia-{elo}.pb.gz'
    if os.path.exists(dest):
        print('have', dest); continue
    url = f'https://github.com/CSSLab/maia-chess/raw/master/maia_weights/maia-{elo}.pb.gz'
    print('downloading', url)
    try:
        urllib.request.urlretrieve(url, dest)
    except Exception as e:
        print('failed:', e)

## 5. Optional: mine a Lichess monthly dump for additional traps

A single month is ~10 GB. The `--max-games` flag caps the scan; 100k
gives a useful trap corpus in ~5 minutes on Colab's CPU.

In [ ]:
# Uncomment to run. Pick a recent month from https://database.lichess.org/
# !pip install -q zstandard
# !python -m data.mine_lichess --month 2024-09 --max-games 100000 --out data/mined_traps.json

## 6. Launch the server

In [ ]:
import subprocess, time, os, signal
# Kill any previous server.
subprocess.run(['pkill', '-f', 'app.server'], check=False)
time.sleep(1)
env = os.environ.copy()
env['PYTHONPATH'] = os.getcwd()
env['CD_PORT'] = '8000'
env['CD_HOST'] = '0.0.0.0'
env['CD_THREADS'] = '4'
server = subprocess.Popen(
    ['python', '-m', 'app.server'],
    env=env, stdout=open('/tmp/cd_server.log', 'w'), stderr=subprocess.STDOUT,
)
print('Server PID:', server.pid)
for _ in range(20):
    time.sleep(0.5)
    try:
        import urllib.request
        urllib.request.urlopen('http://localhost:8000/api/health', timeout=1).read()
        print('Server is up.')
        break
    except Exception:
        continue
else:
    print('Server did not come up — check /tmp/cd_server.log')

## 7. Expose via Cloudflare quick tunnel

No login required; the URL is ephemeral and dies with the kernel.

In [ ]:
import os, subprocess, re, time
if not os.path.exists('cloudflared'):
    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
    !chmod +x cloudflared
subprocess.run(['pkill', '-f', 'cloudflared'], check=False)
time.sleep(1)
tunnel = subprocess.Popen(
    ['./cloudflared', 'tunnel', '--url', 'http://localhost:8000', '--no-autoupdate'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)
public_url = None
for _ in range(60):
    line = tunnel.stdout.readline()
    if not line:
        time.sleep(0.5); continue
    m = re.search(r'(https://[a-z0-9-]+\.trycloudflare\.com)', line)
    if m:
        public_url = m.group(1)
        break
from IPython.display import HTML, display
if public_url:
    display(HTML(f'<h2>ChessDorrie is live → <a href="{public_url}" target="_blank">{public_url}</a></h2>'))
else:
    print('Cloudflared did not return a URL. Check the cell output for errors.')

## Tail the server log (handy if something looks wrong)

In [ ]:
!tail -30 /tmp/cd_server.log